
# Import Libraries


In [27]:
import numpy as np
import sys
import os
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)
    
from decision_tree import DecisionTreeRegressor
from Bagging import BaggingRegressor
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import time


In [28]:
import os
import pandas as pd

#print('cwd:', os.getcwd())
#print('files:', os.listdir())

train_df = pd.read_csv('../data/train.csv')
value_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

def prepare_data(df):
    X = df.drop(columns=['Calories']).values  # Chuyển về numpy array
    y = df['Calories'].values                 # Chuyển về numpy array
    return X, y

X_train, y_train = prepare_data(train_df)
X_val, y_val = prepare_data(value_df)

print(f"Training trên tập dữ liệu kích thước: {X_train.shape}")


Training trên tập dữ liệu kích thước: (9600, 7)


## Tham số tốt nhất: Train 300 cây, độ sâu 15, min_samples_split=2

# Bagging Reggressor from scratch

In [29]:
# Khởi tạo Bagging Regressor
# - n_estimator: số lượng 
# - max_depth: Độ sâu cây tối đa 
model = BaggingRegressor(base_tree_cls=DecisionTreeRegressor, 
                         n_estimators=300, 
                         min_samples_split=2, 
                         max_depth=15)
start=time.time()
model.fit(X_train, y_train)
manual_time=time.time()-start
print(f"Thời gian huấn luyện: {manual_time:.2f} giây")
print("Huấn luyện hoàn tất!")



# Dự đoán trên tập test
y_pred = model.predict(X_val)

# Tính toán sai số
manual_mse = mean_squared_error(y_val, y_pred)
manual_rmse = np.sqrt(manual_mse)
manual_r2 = r2_score(y_val, y_pred)
manual_mae = mean_absolute_error(y_val, y_pred)

print("-" * 30)
print(f"Kết quả đánh giá:")
print(f"RMSE (Sai số trung bình): {manual_rmse:.2f} Calories")
print(f"R2 Score (Độ phù hợp): {manual_r2:.4f}")
print(f"MAE (Sai số tuyệt đối trung bình): {manual_mae:.2f} Calories")
print("-" * 30)



Thời gian huấn luyện: 8332.17 giây
Huấn luyện hoàn tất!
------------------------------
Kết quả đánh giá:
RMSE (Sai số trung bình): 2.93 Calories
R2 Score (Độ phù hợp): 0.9978
MAE (Sai số tuyệt đối trung bình): 1.85 Calories
------------------------------


# Bagging Reggressor using sklearn

# Import Libraries for BaggingRegressor

In [30]:
from sklearn.ensemble import BaggingRegressor as SklearnBaggingRegressor
from sklearn.tree import DecisionTreeRegressor as SklearnDecisionTreeRegressor

In [31]:
'''
model = SklearnBaggingRegressor(estimator=SklearnDecisionTreeRegressor(max_depth=10),
                         n_estimators=20,
                         random_state=42)
'''
my_tree = SklearnDecisionTreeRegressor(
    max_depth=15,            
    min_samples_split=2,   
    random_state=42
)


model = SklearnBaggingRegressor(
    estimator=my_tree,      #
    n_estimators=300,
    random_state=42,
    
)

start=time.time()
model.fit(X_train, y_train)
sklearn_time=time.time()-start
print(f"Thời gian huấn luyện sklearn: {sklearn_time:.2f} giây")
print("Huấn luyện sklearn hoàn tất!")

y_pred = model.predict(X_val)

sklearn_mse = mean_squared_error(y_val, y_pred)
sklearn_rmse = np.sqrt(sklearn_mse)
sklearn_mae = mean_absolute_error(y_val, y_pred)
sklearn_r2 = r2_score(y_val, y_pred)

print("-" * 30)
print(f"Kết quả đánh giá của mô hình lấy từ thư viện:")
print(f"RMSE (Sai số trung bình): {sklearn_rmse:.2f} Calories")
print(f"R2 Score (Độ phù hợp): {sklearn_r2:.4f}")
print(f"MAE (Sai số tuyệt đối trung bình): {sklearn_mae:.2f} Calories")
print("-" * 30)



Thời gian huấn luyện sklearn: 5.82 giây
Huấn luyện sklearn hoàn tất!
------------------------------
Kết quả đánh giá của mô hình lấy từ thư viện:
RMSE (Sai số trung bình): 2.98 Calories
R2 Score (Độ phù hợp): 0.9977
MAE (Sai số tuyệt đối trung bình): 1.85 Calories
------------------------------


In [32]:
results = [
    ["Manual", manual_time, manual_rmse, manual_mae,manual_r2],
    ["Sklearn", sklearn_time, sklearn_rmse, sklearn_mae, sklearn_r2]
]

df = pd.DataFrame(results, columns=["Model Type", "Time (s)", "RMSE", "MAE","R2 Score"])
display(df)

,Model Type,Time (s),RMSE,MAE,R2 Score
0,Manual,8332.167544,2.932184,1.847857,0.997751
1,Sklearn,5.815524,2.975620,1.851843,0.997684
